In [ ]:
# NORTHSTAR -- Tower 32 OAuth3 Governance QA for Solace Browser
# DNA: oauth3(governance) = scope(fine) x ttl(expire) x revoke(instant) x budget(enforce) x stepup(approve) x audit(chain) x consent(explicit)
# Probes OAuth3 scopes, TTL, revocation, budget enforcement, step-up auth, audit, and consent
import os, sys, re, json, hashlib, ast
from pathlib import Path
from datetime import datetime

NORTHSTAR = "oauth3-t32-qa"
NOTEBOOK_PATH = "notebooks/qa/18-oauth3-t32-qa.ipynb"
TOWER = 32
TOWER_NAME = "Tower of OAuth3 Governance"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"
OAUTH3 = SRC / "oauth3"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()
assert OAUTH3.exists(), f"OAuth3 module not found at {OAUTH3}"

In [ ]:
# F1 -- Kenji Watanabe: Scope Definition (CISO)
# Q: Can I define fine-grained scopes? Read vs write distinction?

# Probe: scopes module defines granular permissions
scopes_code = read_file(OAUTH3 / "scopes.py")
has_read_write = bool(re.search(r'read|write|list|delete|create', scopes_code, re.IGNORECASE))
record("F1-P1", "Scopes distinguish read/write/delete actions", has_read_write,
       f"{len(scopes_code)} bytes")

# Probe: scopes use namespace:action format
scope_patterns = re.findall(r'["\']([a-z_]+:[a-z_]+)["\']', scopes_code)
record("F1-P2", "Scopes use namespace:action format",
       len(scope_patterns) >= 2, f"{len(scope_patterns)} scoped patterns: {scope_patterns[:5]}")

# Probe: scope validation/checking logic exists
has_check = bool(re.search(r'check|validate|verify|has_scope|require_scope', scopes_code, re.IGNORECASE))
record("F1-P3", "Scope checking/validation logic exists", has_check)

# Probe: scope test file exists
scope_test = TESTS / "test_scope_display.py"
has_scope_test = scope_test.exists() and scope_test.stat().st_size > 100
record("F1-P4", "Scope display test exists", has_scope_test)

In [ ]:
# F2 -- Fatima Al-Rashid: TTL Enforcement (Security Architect)
# Q: Can tokens expire automatically? No permanent tokens?

# Probe: token module has TTL/expiry logic
token_code = read_file(OAUTH3 / "token.py")
has_ttl = bool(re.search(r'ttl|expir|timeout|lifetime|max_age|valid_until', token_code, re.IGNORECASE))
record("F2-P1", "Token module has TTL/expiry logic", has_ttl,
       f"{len(token_code)} bytes")

# Probe: token expiry is enforced (not just stored)
has_expiry_check = bool(re.search(r'is_expired|check_expir|expired|datetime.*now.*>|time.*>.*expir', token_code, re.IGNORECASE))
record("F2-P2", "Token expiry is actively checked", has_expiry_check)

# Probe: no hardcoded long TTLs (e.g., 365 days)
long_ttl = re.findall(r'ttl\s*=\s*(\d+)', token_code)
long_vals = [int(v) for v in long_ttl if int(v) > 86400 * 30]  # > 30 days in seconds
record("F2-P3", "No excessively long TTL values (>30 days)",
       len(long_vals) == 0, f"Long TTLs: {long_vals}" if long_vals else "all reasonable")

In [ ]:
# F3 -- Henrik Svensson: Instant Revocation (IT Security)
# Q: Can tokens be revoked instantly? Single action?

# Probe: revocation module exists
revocation_code = read_file(OAUTH3 / "revocation.py")
has_revoke = bool(re.search(r'revoke|invalidate|cancel|disable|blacklist', revocation_code, re.IGNORECASE))
record("F3-P1", "Revocation module with invalidation logic exists", has_revoke,
       f"{len(revocation_code)} bytes")

# Probe: revocation is immediate (not batched/delayed)
has_immediate = bool(re.search(r'immediate|instant|real.?time|now|sync', revocation_code, re.IGNORECASE))
no_delay = not bool(re.search(r'batch|delay|queue|later|schedule', revocation_code, re.IGNORECASE))
record("F3-P2", "Revocation appears immediate (no batching)",
       has_immediate or no_delay)

# Probe: revocation test exists
oauth3_tests = list(TESTS.glob("test_oauth3*.py"))
record("F3-P3", "OAuth3 test suite exists", len(oauth3_tests) >= 1,
       f"{len(oauth3_tests)} OAuth3 test files")

In [ ]:
# F4 -- Budget Enforcement
# Q: Are budgets enforced at the token level?

# Probe: budget gates module exists
budget_code = read_file(SRC / "budget_gates.py")
has_budget = bool(re.search(r'budget|cost|limit|threshold|cap|exceed', budget_code, re.IGNORECASE))
record("F4-P1", "Budget gates module enforces cost limits", has_budget,
       f"{len(budget_code)} bytes")

# Probe: budget enforcement raises on exceed (fail-closed)
has_raise_budget = bool(re.search(r'raise.*budget|raise.*limit|raise.*exceed|deny.*budget', budget_code, re.IGNORECASE))
record("F4-P2", "Budget enforcement raises on exceed (fail-closed)", has_raise_budget)

# Probe: budget test exists
budget_test = TESTS / "test_budget_gates.py"
has_budget_test = budget_test.exists() and budget_test.stat().st_size > 100
record("F4-P3", "Budget gates test exists", has_budget_test)

In [ ]:
# F5 -- Step-Up Authentication
# Q: Does step-up auth require re-consent for sensitive ops?

# Probe: step-up module exists
stepup_code = read_file(OAUTH3 / "step_up.py")
has_stepup = bool(re.search(r'step.?up|elevat|re.?auth|re.?consent|sensitive', stepup_code, re.IGNORECASE))
record("F5-P1", "Step-up auth module exists", has_stepup,
       f"{len(stepup_code)} bytes")

# Probe: step-up test exists
stepup_test = TESTS / "test_step_up.py"
has_stepup_test = stepup_test.exists() and stepup_test.stat().st_size > 100
record("F5-P2", "Step-up auth test exists", has_stepup_test)

# Probe: elevated approvals module
elevated_code = read_file(SRC / "approvals" / "elevated.py")
has_elevated = bool(re.search(r'elevat|high.?risk|sensitive|approve', elevated_code, re.IGNORECASE))
record("F5-P3", "Elevated approvals module exists", has_elevated,
       f"{len(elevated_code)} bytes")

In [ ]:
# F6 -- Audit Chain + F7 -- Consent
# Q: Is every OAuth3 action audit-logged with hash chains?
# Q: Is consent explicit and recorded?

# Probe: OAuth3 evidence module for audit logging
oauth3_evidence = read_file(OAUTH3 / "evidence.py")
has_evidence = bool(re.search(r'evidence|audit|log|record|hash|sha256', oauth3_evidence, re.IGNORECASE))
record("F6-P1", "OAuth3 evidence/audit logging module exists", has_evidence,
       f"{len(oauth3_evidence)} bytes")

# Probe: consent UI records explicit consent
consent_code = read_file(OAUTH3 / "consent_ui.py")
has_consent = bool(re.search(r'consent|approve|deny|grant|refuse', consent_code, re.IGNORECASE))
record("F6-P2", "Consent UI records explicit user decisions", has_consent)

# Probe: consent test exists
consent_test = TESTS / "test_consent_ui.py"
has_consent_test = consent_test.exists() and consent_test.stat().st_size > 100
record("F6-P3", "Consent UI test exists", has_consent_test)

# Probe: enforcement module ties all OAuth3 together
enforcement_code = read_file(OAUTH3 / "enforcement.py")
has_enforce = bool(re.search(r'enforce|check|validate|verify|require', enforcement_code, re.IGNORECASE))
record("F6-P4", "OAuth3 enforcement module validates all access", has_enforce,
       f"{len(enforcement_code)} bytes")

In [ ]:
# NEGATIVE SPACE (P16) -- OAuth3 gaps

# Probe: vault uses encryption for stored tokens
vault_code = read_file(OAUTH3 / "vault.py")
has_encrypt = bool(re.search(r'encrypt|aes|gcm|pbkdf|argon|scrypt|cipher|fernet', vault_code, re.IGNORECASE))
record("NS-P1", "Vault uses encryption for token storage", has_encrypt,
       f"{len(vault_code)} bytes")

# Probe: no bare excepts in OAuth3 (Fallback Ban)
oauth3_files = list(OAUTH3.glob("*.py"))
bare_excepts = 0
for of in oauth3_files:
    content = of.read_text(encoding='utf-8', errors='replace')
    bare_excepts += len(re.findall(r'^\s*except\s*:', content, re.MULTILINE))
    bare_excepts += len(re.findall(r'^\s*except\s+Exception\s*:', content, re.MULTILINE))
record("NS-P2", "No bare excepts in OAuth3 modules (Fallback Ban)",
       bare_excepts == 0, f"{bare_excepts} bare excepts across {len(oauth3_files)} files")

# Probe: OAuth3 __init__.py exports key modules
oauth3_init = read_file(OAUTH3 / "__init__.py")
record("NS-P3", "OAuth3 package initialized",
       len(oauth3_init) > 0, f"{len(oauth3_init)} bytes")

# Probe: FS gateway has OAuth3 gates
fs_gate_test = TESTS / "test_fs_gateway_oauth3_gates.py"
has_fs_test = fs_gate_test.exists() and fs_gate_test.stat().st_size > 100
record("NS-P4", "FS gateway OAuth3 gates test exists", has_fs_test)

In [ ]:
# EVIDENCE SUMMARY -- Tower 32 OAuth3 Governance QA
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"

summary = {
    "surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total, "passed": passed, "failed": failed,
    "score": score, "grade": grade, "finding_rate": finding_rate,
    "converged": finding_rate < 5,
    "findings": [r for r in results if r["status"] == "FAIL"],
    "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]
}

print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))